In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import time
!pip install biosppy
import biosppy.signals.ecg as ecg
from scipy import signal
import os
import zipfile
import re
!pip install hrvanalysis
import hrvanalysis

In [ ]:
# Import the CSV file as a pandas DataFrame
df = pd.read_excel('ALL_TRIP_DATA/8/First_by_day_Mod/2021-02-07.xlsx')

In [ ]:
# Convert recordTime column to datetime format
df['recordTime'] = pd.to_datetime(df['recordTime'], format='%Y%m%d%H%M%S')

# Convert datetime format to Unix timestamps
df['unixTime'] = (df['recordTime'] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')

# # Print the resulting dataframe
# print(df.head)

In [ ]:
# FINDING TIME OF OCCURENCE OF ADB FOR TAXI DRIVER 
from datetime import datetime

columns_to_search = ['isAggressiveStreering', 'isHardAcc', 'isHardBrak','isSpeeding']
column_time = ['unixTime']
ADB_Times = []
pd.options.display.float_format = '{:.0f}'.format # nearest whole number for unixTime

for column in columns_to_search:
    true_values = df[df[column] == True]
    if not true_values.empty:
        print(f"True values found in {column}:")
        print(true_values[column_time])

        ADB_Times += true_values['unixTime'].tolist()
    else:
        print(f"No true values found in {column}.")
        
# Convert Unix timestamps to datetime objects and store in a new list
ADB_datetimes = [datetime.fromtimestamp(t) for t in ADB_Times]

for dt in ADB_datetimes:
    print(f"in datetime format: {dt.strftime('%Y-%m-%d %H:%M:%S')}") 

In [ ]:
# OPENING UNIX TIMESTEP FOLDERS AND MAKING EVENLY SPACED ARRAY
start = 1612682795 # has to be input manually for starting folder
end = 1612683995 # again manual for last folder
spacing = 600 # each folder is at 10 minute intervals which is 600s

ECG_timestamp = []
# create array with evenly spaced Unix timestamps for the folders
for i in range(start, end + 1, spacing):
    ECG_timestamp.append(i)
print(ECG_timestamp)

In [ ]:
parent_dir = r"Raw_HR\No_8(1+2)\measure\5a3f3e5d-a090-443d-a3f7-a112c29a7c4f\FilteredECG\250"

# Create an empty list to hold ECG data and time
ecg_data = []

# Specify the desired file names
file_names = ['1612682795.txt', '1612683395.txt', '1612683995.txt']

# Loop over each file
for file_name in file_names:
    file_path = os.path.join(parent_dir, file_name)
    with open(file_path, 'r') as file:
        ecg_raw = file.read()
    # Clean the ECG data
    ecg_clean = re.sub(r'[a-zA-Z]', '', ecg_raw)
    ecg_clean = ecg_clean.replace(",", "")
    ecg_float = np.array([np.float32(x) for x in ecg_clean.split()])
    # Determine the time interval for each data point
    time_duration = 600  # 10-minute interval
    time_step = time_duration / len(ecg_float)
    # Create a time array for the ECG data
    start_time = int(re.search(r'\d+', file_name).group(0))
    time_array = np.linspace(start_time, start_time + time_duration, len(ecg_float))
    # Append the ECG data and time array to the list
    ecg_data.append(np.column_stack((time_array, ecg_float)))

# Concatenate the data from all files into a single array
ecg_data_concat = np.concatenate(ecg_data, axis=0)

# Create a pandas DataFrame with the time and ECG data
df = pd.DataFrame(ecg_data_concat, columns=['time', 'ECG'])
print(len(df['time']))
print(len(df['ECG']))

# Convert Unix timestamps to datetime format
df['time'] = pd.to_datetime(df['time'], unit='s')
print(df)

In [ ]:
print(ADB_datetimes)

In [ ]:
# Find the start and end times for plotting
start_time = ADB_datetimes[1] - pd.Timedelta(minutes=0.5)
end_time = ADB_datetimes[1] + pd.Timedelta(minutes=0.5)

# Filter the DataFrame based on the selected time range
df_range = df[(df['time'] >= start_time) & (df['time'] <= end_time)]
ecg_range = df_range[(df_range['ECG'] >= -0.15) & (df_range['ECG'] <= 0.5)]

plt.figure(figsize=(12, 6))
plt.plot(ecg_range['time'], ecg_range['ECG'], label='ECG signal')
plt.xlabel('Recording time')
plt.ylabel('ECG')
plt.title('Rolling window analysis (Interval: 30s, Rolling step: 1s)')
plt.ylim(-0.2, 0.6)

# Draw a vertical line at the ADB event time
adb_event_time = ADB_datetimes[1]
plt.axvline(x=adb_event_time, color='orange', linestyle='--', linewidth=2, label='ADB Event')

# Add a box around the ADB event
box_start = adb_event_time - pd.Timedelta(seconds=15)
box_end = adb_event_time + pd.Timedelta(seconds=15)
box_patch = mpatches.Patch(facecolor='orange', alpha=0.1, label='ADB Event Box')
plt.axvspan(box_start, box_end, facecolor='orange', alpha=0.1)
handles, labels = plt.gca().get_legend_handles_labels()
handles.append(box_patch)
labels.append('ADB window')

# draw box at ADB
box_start_x = box_start
box_end_x = box_end
box_start_y = -0.19
box_end_y = 0.59

plt.hlines(y=box_start_y, xmin=box_start_x, xmax=box_end_x, color='orange', linewidth=3)
plt.hlines(y=box_end_y, xmin=box_start_x, xmax=box_end_x, color='orange', linewidth=3)
plt.vlines(x=box_start_x, ymin=box_start_y, ymax=box_end_y, color='orange', linewidth=3)
plt.vlines(x=box_end_x, ymin=box_start_y, ymax=box_end_y, color='orange', linewidth=3)

# rollover box
black_box_start = adb_event_time - pd.Timedelta(seconds=14)
black_box_end = adb_event_time + pd.Timedelta(seconds=16)

black_start_x = black_box_start
black_end_x = black_box_end
black_start_y = -0.19
black_end_y = 0.59

rollover_hline = plt.hlines(y=black_start_y, xmin=black_start_x, xmax=black_end_x, color='black', linewidth=3, linestyle='--', label='Rollover Window')
plt.hlines(y=black_start_y, xmin=black_start_x, xmax=black_end_x, color='black', linewidth=3, linestyle='--')
plt.hlines(y=black_end_y, xmin=black_start_x, xmax=black_end_x, color='black', linewidth=3, linestyle='--')
plt.vlines(x=black_start_x, ymin=black_start_y, ymax=black_end_y, color='black', linewidth=3, linestyle='--')
plt.vlines(x=black_end_x, ymin=black_start_y, ymax=black_end_y, color='black', linewidth=3, linestyle='--')
handles.append(rollover_hline)
labels.append('Rollover window')

# Adjust x-axis labels
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
ax.xaxis.set_minor_locator(mdates.SecondLocator(bysecond=range(0, 60, 5)))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.xaxis.set_minor_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.tick_params(axis='x', which='major', rotation=70)
ax.tick_params(axis='x', which='minor', rotation=70)

plt.legend(handles, labels, loc='upper right')  # Increased font size
plt.subplots_adjust(bottom=0.2)
plt.savefig('Rolling_window.jpg', format='jpeg', dpi=300)
plt.show()
# Print the filtered DataFrame
print(df_range)

In [ ]:
# PLOT RR INTERVALS 
out = ecg.ecg(df_range['ECG'], 250, show=False)

# Assume rpeaks is a one-dimensional array of sample indices of the R-peaks
rpeaks = out['rpeaks']
rpeak_times = df_range['time'].iloc[rpeaks]
rr_intervals = np.diff(rpeak_times)/1000000 # this is in ns
print(rpeak_times)
print(rr_intervals)
rr_intervals = rr_intervals.tolist() #################
print(type(rr_intervals))


# Shift rpeak_times array to create an array of times for each RR interval
interval_times = rpeak_times[1:]

# Plot the RR intervals against time
plt.plot(interval_times[:300], rr_intervals[:300])

# Draw a vertical line at the ADB event time
adb_event_time = ADB_datetimes[1]
plt.axvline(x=adb_event_time, color='orange', linestyle='--', linewidth=2, label='ADB Event')

# Adjust x-axis labels
ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
ax.xaxis.set_minor_locator(mdates.SecondLocator(bysecond=range(0, 60, 5)))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.xaxis.set_minor_formatter(mdates.DateFormatter('%H:%M:%S'))
ax.tick_params(axis='x', which='major', rotation=70)
ax.tick_params(axis='x', which='minor', rotation=70)
plt.xlabel('Time')
plt.ylabel('RR Intervals (ms)')
plt.title('RR Intervals for 1 minute')
plt.show()

In [ ]:
# COMPARE RR INTERVALS WITH NN INTERVALS FOR 1 HOUR
nn_intervals = hrvanalysis.preprocessing.get_nn_intervals(rr_intervals, 300, 2000, 'inside', 'both', 'linear')

# Plot the RR intervals against time
plt.plot(interval_times, rr_intervals)
plt.xlabel('Time')
plt.ylabel('RR Intervals (ms)')
plt.title('Raw RR intervals against time')
plt.show()

# Plot the NN intervals against time
plt.plot(interval_times, nn_intervals)
plt.xlabel('Time')
plt.ylabel('NN Intervals (ms)')
plt.title('processed NN intervals against time')
plt.show()

In [ ]:
hrv_data = pd.read_csv('final_data/Final_8.csv')
print(hrv_data)

In [ ]:
start_time = ADB_datetimes[1] - pd.Timedelta(minutes=0.5)
end_time = ADB_datetimes[1] + pd.Timedelta(minutes=0.5)
print(start_time)
print(end_time)
hrv_data['end_time'] = pd.to_datetime(hrv_data['end_time'])
df = hrv_data[(hrv_data['end_time'] > start_time) & (hrv_data['end_time'] < end_time)]
print(df)

In [ ]:
# Create subplots
fig, axs = plt.subplots(2, 2, figsize=(12, 8))

# Plot SDNN data
axs[0, 0].plot(df['end_time'], df['sdnn'], label='SDNN')
axs[0, 0].set_xlabel('Time')
axs[0, 0].set_ylabel('SDNN')
axs[0, 0].set_xticklabels(axs[0, 0].get_xticklabels(), rotation=45)
axs[0, 0].xaxis.set_major_formatter(mdates.DateFormatter(time_format))
axs[0, 0].axvline(x=adb_event_time, color='red', linestyle='--', linewidth=2, label='ADB Event')
axs[0, 0].legend()

# Plot RMSSD data
axs[0, 1].plot(df['end_time'], df['rmssd'], label='RMSSD')
axs[0, 1].set_xlabel('Time')
axs[0, 1].set_ylabel('RMSSD (ms)')
axs[0, 1].set_xticklabels(axs[0, 1].get_xticklabels(), rotation=45)
axs[0, 1].xaxis.set_major_formatter(mdates.DateFormatter(time_format))
axs[0, 1].axvline(x=adb_event_time, color='red', linestyle='--', linewidth=2, label='ADB Event')
axs[0, 1].legend()

# Plot nHF data
window_size = 10
smoothed_nhf = uniform_filter1d(df['hfnu'], size=window_size)
axs[1, 0].plot(df['end_time'], smoothed_nhf, label='nHF')
axs[1, 0].set_xlabel('Time')
axs[1, 0].set_ylabel('nHF (%)')
axs[1, 0].set_xticklabels(axs[1, 0].get_xticklabels(), rotation=45)
axs[1, 0].xaxis.set_major_formatter(mdates.DateFormatter(time_format))
axs[1, 0].axvline(x=adb_event_time, color='red', linestyle='--', linewidth=2, label='ADB Event')
axs[1, 0].legend()

# Plot nLF data
window_size = 10
smoothed_nlf = uniform_filter1d(df['lfnu'], size=window_size)
axs[1, 1].plot(df['end_time'], smoothed_nlf, label='nLF')
axs[1, 1].set_xlabel('Time')
axs[1, 1].set_ylabel('nLF (%)')
axs[1, 1].set_xticklabels(axs[1, 1].get_xticklabels(), rotation=45)
axs[1, 1].xaxis.set_major_formatter(mdates.DateFormatter(time_format))
axs[1, 1].axvline(x=adb_event_time, color='red', linestyle='--', linewidth=2, label='ADB Event')
axs[1, 1].legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig.suptitle('HRV Parameters Comparison', fontsize=16)

plt.savefig('hrv_plots.jpg', format='jpeg')
plt.show()
